# Laws-DE Token-Budget Investigation

**Goal:** decide whether the current single-config run (`max_text_chars=1500`, `max_new_tokens=450`, `max_model_len=3072`, `max_num_seqs=384`) is wasting GPU memory / wall-clock for the laws_de corpus, and if so, propose a **bucketed** config (e.g. 4–17 buckets) that fits each length regime tightly.

## Why this matters

The corpus summary already tells us laws_de is heavily **left-skewed**:

```
<100 chars   :  29,439 rows  (17.0%)
100-299 chars:  105,171 rows (60.8%)
300-599 chars:  29,776 rows (17.2%)
600-1499     :   7,440 rows  (4.3%)
>=1500       :   1,207 rows  (0.7%)
```

Two structural inefficiencies in the current notebook:

1. **Every request reserves the same `max_new_tokens=450`.** A 50-char article almost certainly produces a JSON object with empty rule fields and 1–2 terms — well under 200 output tokens. The 450-token ceiling rarely binds, but vLLM's scheduler still sizes its admission decisions around `max_model_len` (prompt cap + output cap), so a tighter ceiling for the short majority lets us pack more concurrent slots and fail-fast on degenerate runaway outputs sooner.
2. **`max_text_chars=1500` is a global hard truncation.** Only 0.7% of rows hit it, but those rows lose the tail half of the article (`trim_text` keeps head + tail). For the long-tail bucket we'd rather pay the prompt-token cost once than silently drop legal context.

## What this notebook produces

1. Empirical **prompt-token distribution** for the full 173k-row JSONL using the *real* Qwen3-8B tokenizer and the *real* `render_prompt` chain.
2. Per-bucket recommendations: `max_text_chars`, `max_model_len`, `max_new_tokens`, `max_num_seqs`.
3. KV-budget arithmetic: how many concurrent slots a smaller `max_model_len` actually buys on the RTX PRO 6000 (95 GB) target.
4. **Optional GPU pilot** (~2k stratified rows): measures *real* output-token p99 per bucket so the `max_new_tokens` recommendation is empirical, not heuristic. Skipped automatically when no GPU is present.
5. A **single JSON config** written to `artifacts/laws_de_bucketed_config.json` that the production notebook can ingest.

## What this notebook does NOT do

- It does not run the full enrichment.
- It does not modify the production notebook. Once we agree on the bucket plan, that's a separate edit.
- It does not assume the laws_de schema produces the same output-token distribution as the court schema (different field set, different prompt). We measure on the laws prompt directly.

## Cell 1 — Setup & paths

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- locate the law_llm_input.jsonl ---
IN_KAGGLE = bool(os.environ.get('KAGGLE_URL_BASE') or Path('/kaggle').exists())
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/swiss_law')
elif IN_KAGGLE:
    BASE_DIR = Path('/kaggle/working/swiss_law')
else:
    # Local Windows path (project root). Override here if your layout differs.
    BASE_DIR = Path(r'e:\\swiss_citation_extraction')

ARTIFACTS_DIR = BASE_DIR / 'artifacts'
DATA_DIR = BASE_DIR / 'data'

# Candidate locations for the input — first existing wins.
INPUT_CANDIDATES = [
    ARTIFACTS_DIR / 'law_llm_input.jsonl',
    DATA_DIR / 'law_llm_input.jsonl',
    Path('/kaggle/input') / 'law_llm_input.jsonl',
    Path.cwd() / 'law_llm_input.jsonl',
]
INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError(
        'Could not locate law_llm_input.jsonl. Edit the BASE_DIR in this cell, '
        'or copy the file into one of:\n  ' + '\n  '.join(str(p) for p in INPUT_CANDIDATES)
    )

OUTPUT_DIR = ARTIFACTS_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_OUT = OUTPUT_DIR / 'laws_de_bucketed_config.json'

print('Environment :', 'kaggle' if IN_KAGGLE else 'colab' if IN_COLAB else 'local')
print('Input JSONL :', INPUT_PATH, f'({INPUT_PATH.stat().st_size / 1024**2:.1f} MB)')
print('Output dir  :', OUTPUT_DIR)

## Cell 2 — Reproduce the production prompt builder verbatim

Token counts are only meaningful if we tokenize **exactly what the production notebook sends to vLLM**. The system prompt below is byte-identical to `enrich_laws_de_qwen3_8b_kaggle.ipynb` cell 4. Update this cell if the production prompt ever changes.

In [ ]:
LLM_SCHEMA_HINT = {
    'english_summary': '<=2 sentences English; what THIS article says (not the law title)',
    'legal_rule': '<=25 words operative rule; empty for boilerplate',
    'applicability_conditions': ['0-5 short English conditions'],
    'exceptions_or_limitations': ['0-5 English carve-outs'],
    'legal_question': '<=18 words; empty for boilerplate',
    'concepts_en': ['3-8 English legal concepts'],
    'terms_de_to_en': [
        {'de': 'first verbatim DE/FR/IT term', 'en': 'Swiss-legal English equivalent'},
        {'de': 'second verbatim term',         'en': 'Swiss-legal English equivalent'},
    ],
    'defined_terms': [
        {'term': 'first verbatim term defined here', 'definition': 'English gloss'},
        {'term': 'second verbatim term',             'definition': 'English gloss'},
    ],
    'addressees': ['0-6 English labels: who is bound'],
    'sanctions_or_consequences': ['0-4 English items in text'],
    'provision_role_llm': 'definition|purpose|scope|principle|right_or_entitlement|duty|prohibition|procedure|competence|sanction_or_penalty|data_reporting|fees_or_costs|transitional_or_commencement|other',
    'specificity_score': '0..1',
}
_SCHEMA_JSON = json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False, separators=(',', ':'))

SYSTEM_PROMPT = f'''You are a Swiss legal-interpretation assistant working on individual articles
of Swiss federal law (Bundesgesetze / lois fédérales / leggi federali, Verordnungen /
ordonnances / ordinanze, the Bundesverfassung / Constitution fédérale / Costituzione
federale, Verträge / traités / trattati, SR-numbered statutes). The article text is
in German, French, or Italian. Your job is to extract the operative legal meaning
into English, preserving the exact original-language legal terms.

Return exactly one compact JSON object with exactly this shape (all keys present):
{_SCHEMA_JSON}

Hard rules:
1. JSON only. No prose, no markdown, no preamble.
2. Use English for ALL semantic fields except `terms_de_to_en[].de` and
   `defined_terms[].term`, which MUST be exact substrings of the source text in
   its ORIGINAL language (DE, FR, or IT). The JSON key stays `de` regardless of
   the source language — it just means "original-language term".
3. Map each original-language term to its **Swiss-legal English equivalent**,
   NOT a literal translation. Use the row matching the source language; never
   mix languages within one term entry.

   German (DE) -> English:
     Bewilligung -> permit / authorisation
     Verfügung -> formal administrative order
     Rechtsbegehren -> prayer for relief
     Zuständigkeit -> jurisdiction / competence
     Aufsichtsbehörde -> supervisory authority
     Inverkehrbringen -> placing on the market
     Tatbestand -> set of facts / elements of the offence
     Beschwerde -> appeal
     Eidgenössisch -> federal (Swiss)
     Bundesrat -> Federal Council

   French (FR) -> English:
     autorisation -> permit / authorisation
     décision -> formal administrative order
     conclusions -> prayer for relief
     compétence -> jurisdiction / competence
     autorité de surveillance -> supervisory authority
     mise sur le marché -> placing on the market
     état de fait -> set of facts / elements of the offence
     recours -> appeal
     fédéral -> federal (Swiss)
     Conseil fédéral -> Federal Council

   Italian (IT) -> English:
     autorizzazione -> permit / authorisation
     decisione -> formal administrative order
     conclusioni -> prayer for relief
     competenza -> jurisdiction / competence
     autorità di vigilanza -> supervisory authority
     immissione in commercio -> placing on the market
     fattispecie -> set of facts / elements of the offence
     ricorso -> appeal
     federale -> federal (Swiss)
     Consiglio federale -> Federal Council

   Acronyms (language-invariant — keep verbatim): EDI / DFI / SBFI / SEFRI /
   FINMA / ESTV / AFC / EJPD / DFJP / DEFR / IFSN / FF / RS / SR.

   When in doubt, prefer EU/UK statutory English over US wording.
3a. EACH term MUST be its OWN object inside the array. Never write
    {{"de":"A","de":"B"}} or {{"de":"A":"B"}}. Correct shape is
    [{{"de":"A","en":"...A"}},{{"de":"B","en":"...B"}}].
    Same rule for defined_terms with `term`/`definition`.
4. Do NOT invent statute citations, BGE numbers, dates, party names, or
   sanctions that are not literally in the text.
5. Do NOT translate literally. If the source is metaphorical or formal, pick
   the recognised legal English term.
6. Never copy the law title into `english_summary`. Describe what THIS article
   says, not what the parent statute is about.
7. For repeal markers ("Aufgehoben" / "Abrogé" / "Abrogato"), commencement
   clauses ("Tritt am ... in Kraft" / "Entre en vigueur le ..." / "Entra in
   vigore il ..."), pure fee tables, annex code lists, or transitional
   provisions:
     - keep `legal_rule`, `applicability_conditions`, `exceptions_or_limitations`,
       `legal_question` empty.
     - still fill `english_summary`, `concepts_en`, `terms_de_to_en`,
       `provision_role_llm`, `specificity_score` (low).
8. Never put `Art.`, `Abs.`, `Buchstabe`, `Ziffer`, `al.`, `let.`, `ch.`,
   `cpv.`, `lett.`, `n.`, or SR numbers into `terms_de_to_en` — those are
   anchors, not legal terms.
9. `addressees` MUST use labels from this controlled vocabulary (pick the
   closest match; pick multiple when the article binds several actors). Only
   add a free-form label when NONE of these fits:
     - Federal Council
     - federal department or office
     - supervisory authority
     - competent cantonal authority
     - competent communal authority
     - court
     - public prosecutor
     - natural person
     - legal entity / undertaking
     - employer
     - employee
     - taxpayer
     - data subject
     - data controller or processor
     - market participant / operator
     - service provider
     - consumer / customer
     - foreign authority or international organisation
10. Cap: 10 terms_de_to_en, 4 defined_terms, 6 addressees, 5 conditions, 5
    exceptions, 8 concepts_en. Trim to the most salient.
11. If a field has nothing to populate, return an empty string or empty list,
    but the key MUST be present. JSON only.'''

USER_TEMPLATE = '''Citation: {citation}
Law title: {law_title}
Section path: {section_path}
Law code: {law_code}   Article: {article}   Units: {units}
Source type: {source_type}   Enactment year: {year}
Static legal area hint: {legal_area_hint}
Static domain hints: {domain_hints}

Article text (verbatim, original language):
"""
{text}
"""

Return JSON only.'''


def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if max_chars <= 0 or len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()


def _coerce_str(x: Any, max_chars: int = 0) -> str:
    if x is None:
        return ''
    if isinstance(x, str):
        s = x
    elif isinstance(x, (list, tuple)):
        s = ', '.join(_coerce_str(i) for i in x if i)
    elif isinstance(x, dict):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    s = s.strip()
    return s[:max_chars] if max_chars else s


def _units_to_str(units: Any) -> str:
    if not units:
        return ''
    if isinstance(units, str):
        return units
    if not isinstance(units, (list, tuple)):
        return str(units)
    parts = []
    for u in units:
        if u is None:
            continue
        if isinstance(u, str):
            s = u.strip()
        elif isinstance(u, dict):
            label = (u.get('label') or u.get('text') or '').strip()
            if not label:
                head = (u.get('marker') or u.get('unit') or u.get('type') or u.get('kind') or '').strip()
                tail = u.get('value', u.get('number', u.get('n', '')))
                tail = '' if tail is None else str(tail).strip()
                label = (head + ' ' + tail).strip() if (head or tail) else json.dumps(u, ensure_ascii=False)
            s = label
        else:
            s = str(u).strip()
        if s:
            parts.append(s)
    return ', '.join(parts)


def build_user_prompt(row: dict, max_text_chars: int) -> str:
    structural = row.get('structural') or {}
    title_meta = row.get('title_metadata') or {}
    hints = row.get('static_hints') or {}
    enactment_date = _coerce_str(title_meta.get('enactment_date'))
    year = _coerce_str(title_meta.get('enactment_year')) or (enactment_date[:4] if enactment_date else '')
    return USER_TEMPLATE.format(
        citation=_coerce_str(row.get('citation'), 200),
        law_title=_coerce_str(row.get('law_title'), 300),
        section_path=_coerce_str(row.get('title_section_path'), 200),
        law_code=_coerce_str(structural.get('law_code'), 60),
        article=_coerce_str(structural.get('article'), 40),
        units=_units_to_str(structural.get('units')),
        source_type=_coerce_str(title_meta.get('source_type'), 60),
        year=year,
        legal_area_hint=_coerce_str(hints.get('legal_area_static'), 100),
        domain_hints=', '.join(_coerce_str(d, 80) for d in (hints.get('domain_labels_en') or [])[:5] if d),
        text=trim_text(row.get('text', ''), max_text_chars),
    )

print('Prompt builder ready.')
print(f'  SYSTEM_PROMPT chars: {len(SYSTEM_PROMPT):,}')
print(f'  schema-hint chars  : {len(_SCHEMA_JSON):,}')

## Cell 3 — Load all rows from JSONL

In [ ]:
MIN_TEXT_CHARS = 20  # matches the production filter

rows: list[dict] = []
skipped_no_text = 0
skipped_short = 0
with INPUT_PATH.open(encoding='utf-8') as f:
    for line_num, line in enumerate(tqdm(f, desc='read jsonl'), 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            continue
        text = (rec.get('text') or '').strip()
        if not (rec.get('citation') or '').strip() or not text:
            skipped_no_text += 1
            continue
        if len(text) < MIN_TEXT_CHARS:
            skipped_short += 1
            continue
        rows.append(rec)

df = pd.DataFrame(rows)
df['text_chars'] = df['text'].astype(str).str.len()

print(f'\nLoaded   : {len(df):,} rows')
print(f'Skipped  : {skipped_no_text:,} no_text, {skipped_short:,} short')
print('\nText-char quantiles (raw, before truncation):')
for q in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999, 1.0]:
    print(f'  p{int(q*1000)/10:>5}: {int(df["text_chars"].quantile(q)):>6,}')
print(f'\nRows ≥1500 chars (currently truncated): {(df["text_chars"] >= 1500).sum():,}'
      f' ({(df["text_chars"] >= 1500).mean()*100:.2f}%)')
print(f'Rows ≥3000 chars                       : {(df["text_chars"] >= 3000).sum():,}'
      f' ({(df["text_chars"] >= 3000).mean()*100:.2f}%)')
print(f'Rows ≥6000 chars                       : {(df["text_chars"] >= 6000).sum():,}'
      f' ({(df["text_chars"] >= 6000).mean()*100:.2f}%)')

## Cell 4 — Tokenize prompts with the real Qwen3-8B tokenizer

We render the **full chat template** (system + user + assistant generation prompt), then tokenize. This is byte-equivalent to what vLLM sees.

By default we tokenize a **stratified sample of 10,000 rows** (≈30 s on Kaggle CPU). Set `TOKENIZE_ALL = True` to tokenize all 173k rows (~5 min). Sample is plenty for distribution shape; full pass only matters if you want exact counts per long-tail bucket.

In [ ]:
# --- knobs ---
TOKENIZE_ALL = False       # set True for exact full-corpus token counts (~5 min)
SAMPLE_SIZE = 10_000        # used when TOKENIZE_ALL=False; stratified across char buckets
MEASURE_BOTH = True         # measure both (a) current truncated prompts and (b) untruncated
MAX_TEXT_CHARS_CURRENT = 1500  # production setting today
MAX_TEXT_CHARS_NOTRUNC = 0     # 0 = no truncation; the upper bound
MODEL_NAME = 'Qwen/Qwen3-8B-AWQ'

try:
    from transformers import AutoTokenizer  # noqa: F401
except Exception:
    print('transformers not installed; installing...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51.0'])
    from transformers import AutoTokenizer

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f'Tokenizer loaded: {MODEL_NAME} (fast={tok.is_fast})')

# Cache the system-prefix length once (fixed for the whole run, prefix-cached in vLLM).
_probe_msgs = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'X'},
]
try:
    _probe_text = tok.apply_chat_template(
        _probe_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
except TypeError:
    _probe_text = tok.apply_chat_template(_probe_msgs, tokenize=False, add_generation_prompt=True)
PROBE_LEN = len(tok(_probe_text, add_special_tokens=False)['input_ids'])
print(f'Chat-template scaffold + system+schema prefix: {PROBE_LEN} tokens (prefix-cached).')


def render_chat(row: dict, max_text_chars: int) -> str:
    user = build_user_prompt(row, max_text_chars=max_text_chars)
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def tokenize_prompts(df_in: pd.DataFrame, max_text_chars: int, batch_size: int = 256) -> np.ndarray:
    out = np.zeros(len(df_in), dtype=np.int32)
    cursor = 0
    for i in tqdm(range(0, len(df_in), batch_size), desc=f'tokenize (max_chars={max_text_chars})'):
        chunk = df_in.iloc[i:i + batch_size]
        prompts = [render_chat(r._asdict() if hasattr(r, '_asdict') else r, max_text_chars)
                   for r in chunk.to_dict('records')]
        enc = tok(prompts, add_special_tokens=False)
        out[cursor:cursor + len(prompts)] = [len(ids) for ids in enc['input_ids']]
        cursor += len(prompts)
    return out


if TOKENIZE_ALL:
    sample_df = df.reset_index(drop=True)
else:
    # Stratified sample by char-length bucket so the long tail is well represented.
    edges = [0, 100, 300, 600, 1500, 3000, 10**9]
    df = df.assign(_char_bucket=pd.cut(df['text_chars'], bins=edges, right=False, include_lowest=True))
    rng = np.random.default_rng(42)
    parts = []
    target_per_bucket = SAMPLE_SIZE // (len(edges) - 1)
    for b, sub in df.groupby('_char_bucket', observed=True):
        n = min(target_per_bucket, len(sub))
        idx = rng.choice(len(sub), size=n, replace=False)
        parts.append(sub.iloc[idx])
    sample_df = pd.concat(parts, ignore_index=True)
    print(f'Stratified sample: {len(sample_df):,} rows ({len(edges)-1} char buckets)')

tok_current = tokenize_prompts(sample_df, max_text_chars=MAX_TEXT_CHARS_CURRENT)
sample_df['prompt_tok_current'] = tok_current

if MEASURE_BOTH:
    tok_notrunc = tokenize_prompts(sample_df, max_text_chars=MAX_TEXT_CHARS_NOTRUNC)
    sample_df['prompt_tok_notrunc'] = tok_notrunc

print('\n--- Prompt-token quantiles (CURRENT, max_text_chars=1500) ---')
for q in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999, 1.0]:
    print(f'  p{q*100:>6.2f}: {int(np.quantile(tok_current, q)):>5}')
if MEASURE_BOTH:
    print('\n--- Prompt-token quantiles (NO TRUNCATION) ---')
    for q in [0.5, 0.75, 0.9, 0.95, 0.99, 0.999, 1.0]:
        print(f'  p{q*100:>6.2f}: {int(np.quantile(tok_notrunc, q)):>5}')
    extra = tok_notrunc - tok_current
    print(f'\nRemoving truncation costs: median +{int(np.median(extra))} tok, '
          f'p99 +{int(np.quantile(extra, 0.99))} tok, max +{int(extra.max())} tok per row.')

## Cell 5 — Plot the distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(tok_current, bins=80, color='#3366cc', alpha=0.85)
axes[0].axvline(3072, color='red', ls='--', label='current max_model_len=3072')
axes[0].set_title('Prompt token length (current, max_text_chars=1500)')
axes[0].set_xlabel('tokens')
axes[0].set_ylabel('rows')
axes[0].legend()

axes[1].hist(np.log10(np.maximum(tok_current, 1)), bins=80, color='#3366cc', alpha=0.85)
axes[1].set_title('Prompt token length (log10)')
axes[1].set_xlabel('log10 tokens')
axes[1].set_ylabel('rows')
fig.tight_layout()
plt.show()

## Cell 6 — Bucketing strategy

The user's intuition is to split into ~17 buckets. Pure quantile-based splitting on prompt-token length will collapse buckets that share the same recommended `max_model_len` (after rounding to a sensible block size — vLLM's KV block is 16 tokens, but graph capture sizes are coarser, so we round to 256).

We compute the quantile-based bucket plan first, then **merge consecutive buckets** whose recommended `max_model_len` rounds to the same value. This gives the smallest practical bucket count that still captures the length variance.

In [ ]:
# --- knobs ---
NUM_BUCKETS_RAW = 17        # initial quantile slices
PROMPT_TOK_SOURCE = 'notrunc' if MEASURE_BOTH else 'current'  # 'notrunc' = ignore truncation when planning
OUTPUT_TOK_BUDGET_DEFAULT = 450  # used when no pilot has been run
MAX_MODEL_LEN_ROUND = 256   # round to nearest multiple
MAX_MODEL_LEN_HEADROOM = 64  # extra slack on top of (prompt_p99 + output_budget)

# Step 1 — derive bucket edges from quantiles of the chosen prompt-token series.
tok_full = sample_df[f'prompt_tok_{PROMPT_TOK_SOURCE}'].values
qs = np.linspace(0, 1, NUM_BUCKETS_RAW + 1)
edges = np.unique(np.quantile(tok_full, qs).astype(int))
edges[0] = max(edges[0] - 1, 0)
edges[-1] = edges[-1] + 1
print(f'Initial bucket edges ({len(edges)-1} buckets):', edges.tolist())

# Step 2 — assign rows to buckets and compute per-bucket stats.
def round_up(x: int, base: int) -> int:
    return int(np.ceil(x / base) * base)

buckets = []
for i in range(len(edges) - 1):
    mask = (tok_full >= edges[i]) & (tok_full < edges[i + 1])
    if not mask.any():
        continue
    sub = tok_full[mask]
    p50 = int(np.quantile(sub, 0.50))
    p99 = int(np.quantile(sub, 0.99))
    pmax = int(sub.max())
    rec_mml = round_up(p99 + OUTPUT_TOK_BUDGET_DEFAULT + MAX_MODEL_LEN_HEADROOM, MAX_MODEL_LEN_ROUND)
    buckets.append({
        'lo': int(edges[i]), 'hi': int(edges[i + 1]),
        'count': int(mask.sum()),
        'prompt_p50': p50, 'prompt_p99': p99, 'prompt_max': pmax,
        'rec_max_model_len': rec_mml,
    })

# Step 3 — merge consecutive buckets whose rec_max_model_len matches.
merged = []
for b in buckets:
    if merged and merged[-1]['rec_max_model_len'] == b['rec_max_model_len']:
        m = merged[-1]
        m['hi'] = b['hi']
        m['count'] += b['count']
        m['prompt_p99'] = max(m['prompt_p99'], b['prompt_p99'])
        m['prompt_max'] = max(m['prompt_max'], b['prompt_max'])
        m['prompt_p50'] = (m['prompt_p50'] + b['prompt_p50']) // 2
    else:
        merged.append(dict(b))

print(f'\nAfter merging consecutive buckets with same rec_max_model_len: {len(merged)} buckets')
print(pd.DataFrame(merged))

## Cell 7 — KV-cache budget → recommended `max_num_seqs` per bucket

Per-token KV memory for **Qwen3-8B** (32 layers, 4 KV heads, 128 head dim):

    K + V tensors per token = 2 × num_layers × num_kv_heads × head_dim × dtype_bytes
                            = 2 × 32 × 4 × 128 × bytes
                            = 32,768 × bytes

- fp16 KV: 65,536 bytes/token (64 KiB)
- fp8 KV : 32,768 bytes/token (32 KiB) — used when `cfg.kv_cache_dtype='fp8'`

Available KV pool ≈ `gpu_memory_utilization × VRAM − model_weights − activation_overhead`. For Qwen3-8B-AWQ at 0.92 mem util on a 95 GB Blackwell, vLLM typically reports ~80 GB usable for KV after weights (~6 GB) and CUDA-graph buffers.

In [ ]:
# --- knobs ---
GPU_VRAM_GB = 95.0          # RTX PRO 6000 Blackwell on Kaggle
GPU_MEM_UTIL = 0.92
MODEL_WEIGHTS_GB = 6.0       # Qwen3-8B-AWQ ~6 GB (4-bit)
ACTIVATION_OVERHEAD_GB = 4.0 # CUDA graphs + scratch
KV_DTYPE_BYTES = 2           # 2 = fp16, 1 = fp8
PER_TOKEN_KV_BYTES = 2 * 32 * 4 * 128 * KV_DTYPE_BYTES  # K+V × layers × heads × dim × bytes
PREFIX_CACHED_TOKENS = PROBE_LEN  # system+schema; shared across all sequences

kv_pool_bytes = (GPU_VRAM_GB * GPU_MEM_UTIL - MODEL_WEIGHTS_GB - ACTIVATION_OVERHEAD_GB) * 1024**3
kv_pool_bytes -= PREFIX_CACHED_TOKENS * PER_TOKEN_KV_BYTES  # the prefix lives in the pool too
kv_pool_tokens = int(kv_pool_bytes // PER_TOKEN_KV_BYTES)
print(f'Estimated KV pool: {kv_pool_bytes/1024**3:.1f} GiB '
      f'= {kv_pool_tokens:,} tokens at {KV_DTYPE_BYTES}-byte KV')
print(f'Per-slot KV cost at max_model_len=3072: {3072 * PER_TOKEN_KV_BYTES / 1024**2:.1f} MiB')
print(f'Theoretical max slots at current config (3072): {kv_pool_tokens // 3072:,}')
print()

# Hard upper bound on max_num_seqs — vLLM's scheduler quality drops past ~1024.
MAX_NUM_SEQS_CAP = 1024

for b in merged:
    # Each slot's worst case = bucket prompt_p99 + output_budget. Use that for sizing.
    per_slot_tokens = b['rec_max_model_len']
    raw_slots = kv_pool_tokens // per_slot_tokens
    b['rec_max_num_seqs'] = int(min(raw_slots, MAX_NUM_SEQS_CAP))
    b['per_slot_kv_mib'] = round(per_slot_tokens * PER_TOKEN_KV_BYTES / 1024**2, 2)

print(pd.DataFrame(merged)[['lo','hi','count','prompt_p99','rec_max_model_len','per_slot_kv_mib','rec_max_num_seqs']].to_string(index=False))

## Cell 8 — Per-bucket recommended `max_text_chars`

We invert the prompt-token-vs-text-chars relationship empirically. For each bucket we want a `max_text_chars` cap such that the bucket's prompt-token p99 stays inside its `max_model_len` even after subtracting the output budget.

In [ ]:
# Build a (text_chars → prompt_tokens) map from the sample (use the no-trunc series)
src_col = 'prompt_tok_notrunc' if MEASURE_BOTH else 'prompt_tok_current'
ratio = (sample_df[src_col] - PROBE_LEN).clip(lower=1) / sample_df['text_chars'].clip(lower=1)
median_tok_per_char = float(np.median(ratio))
p99_tok_per_char = float(np.quantile(ratio, 0.99))
print(f'Empirical tok/char ratio: median={median_tok_per_char:.3f}, p99={p99_tok_per_char:.3f}')
print(f'(So a 1500-char article costs ~{int(1500*median_tok_per_char)} median, '
      f'~{int(1500*p99_tok_per_char)} p99 prompt tokens.)\n')

# Within each bucket, max_text_chars = the highest text_chars value present.
# We need to map bucket boundaries (which are in tokens) back to char limits.
# Easiest: for each bucket, look at the sample rows that landed in it and take their text_chars p99.
tok_series = sample_df[src_col].values
char_series = sample_df['text_chars'].values

for b in merged:
    mask = (tok_series >= b['lo']) & (tok_series < b['hi'])
    if mask.any():
        chars_in_bucket = char_series[mask]
        b['text_p50'] = int(np.quantile(chars_in_bucket, 0.50))
        b['text_p99'] = int(np.quantile(chars_in_bucket, 0.99))
        b['text_max'] = int(chars_in_bucket.max())
        # Recommendation: cap at p99 of the bucket, rounded up to nearest 100.
        # For the last (longest) bucket, we set NO truncation — it has very few rows.
        if b is merged[-1]:
            b['rec_max_text_chars'] = 0  # 0 = no truncation
        else:
            b['rec_max_text_chars'] = int(np.ceil(b['text_p99'] / 100) * 100)
    else:
        b['text_p50'] = b['text_p99'] = b['text_max'] = 0
        b['rec_max_text_chars'] = 1500

print(pd.DataFrame(merged)[['lo','hi','count','text_p50','text_p99','text_max','rec_max_text_chars']].to_string(index=False))

## Cell 9 — Optional GPU pilot to measure real output-token p99 per bucket

Skipped automatically when no GPU is available or `vllm` isn't installed. When run, it samples ~120 rows per bucket, generates with the full production sampling params, and records actual output-token counts. The recommended `max_new_tokens` per bucket is then `output_p99 + headroom` rather than the global default of 450.

This is the **single most valuable cell in the notebook** — every other recommendation is just arithmetic from prompt-token counts, but only an empirical pilot tells us how output length actually correlates with input length on this specific schema and prompt.

In [ ]:
# --- knobs ---
RUN_GPU_PILOT = True            # set False to skip even when GPU is present
PILOT_ROWS_PER_BUCKET = 120     # 120 × ~10 buckets ≈ 1200 generations, < 5 min on the target GPU
PILOT_TEMPERATURE = 0.1
PILOT_TOP_P = 0.9
PILOT_HARD_MAX_TOKENS = 700     # ceiling for the pilot only (so we capture the true p99 even if it exceeds 450)
PILOT_OUTPUT_HEADROOM = 80      # added on top of measured p99 for the recommendation

GPU_AVAILABLE = False
try:
    import torch
    GPU_AVAILABLE = bool(torch.cuda.is_available())
except Exception:
    pass

PILOT_RAN = False
if RUN_GPU_PILOT and GPU_AVAILABLE:
    try:
        from vllm import LLM, SamplingParams
    except Exception as exc:
        print(f'vLLM not importable ({exc!r}); skipping pilot.')
    else:
        # Smallest config that fits the longest prompt in the corpus + 700-token output budget.
        pilot_max_model_len = round_up(int(np.max(tok_full)) + PILOT_HARD_MAX_TOKENS + 64, 256)
        print(f'Loading vLLM pilot engine: max_model_len={pilot_max_model_len}, max_num_seqs=128')
        llm = LLM(
            model=MODEL_NAME,
            trust_remote_code=True,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.85,
            max_model_len=pilot_max_model_len,
            max_num_seqs=128,
            quantization='awq_marlin',
            enable_prefix_caching=True,
            disable_log_stats=True,
        )
        params = SamplingParams(
            temperature=PILOT_TEMPERATURE, top_p=PILOT_TOP_P,
            max_tokens=PILOT_HARD_MAX_TOKENS, repetition_penalty=1.0,
        )
        # Sample rows per bucket from the pre-tokenized sample_df.
        rng = np.random.default_rng(0)
        for b in merged:
            mask = (tok_series >= b['lo']) & (tok_series < b['hi'])
            idx = np.where(mask)[0]
            if len(idx) == 0:
                b['out_p50'] = b['out_p99'] = b['out_max'] = 0
                b['rec_max_new_tokens'] = OUTPUT_TOK_BUDGET_DEFAULT
                continue
            pick = rng.choice(idx, size=min(PILOT_ROWS_PER_BUCKET, len(idx)), replace=False)
            picked = sample_df.iloc[pick].to_dict('records')
            prompts = [render_chat(r, max_text_chars=b['rec_max_text_chars']) for r in picked]
            t0 = time.time()
            outs = llm.generate(prompts, sampling_params=params, use_tqdm=False)
            lens = [len(o.outputs[0].token_ids) if o.outputs else 0 for o in outs]
            elapsed = time.time() - t0
            lens = np.array(lens, dtype=np.int32)
            b['out_p50'] = int(np.quantile(lens, 0.50))
            b['out_p99'] = int(np.quantile(lens, 0.99))
            b['out_max'] = int(lens.max())
            b['rec_max_new_tokens'] = int(np.ceil((b['out_p99'] + PILOT_OUTPUT_HEADROOM) / 32) * 32)
            print(f"  bucket [{b['lo']:>4}, {b['hi']:>5}) n={len(prompts):>3}: "
                  f"out p50={b['out_p50']:>3}, p99={b['out_p99']:>3}, max={b['out_max']:>3} "
                  f"→ rec_max_new_tokens={b['rec_max_new_tokens']:>3}  ({elapsed:.1f}s)")
        PILOT_RAN = True
        # Free GPU memory so downstream cells can re-load if they want.
        del llm
        import gc; gc.collect(); torch.cuda.empty_cache()
        print('\nPilot complete; vLLM engine released.')
else:
    print(f'Pilot skipped (RUN_GPU_PILOT={RUN_GPU_PILOT}, GPU_AVAILABLE={GPU_AVAILABLE}).')

if not PILOT_RAN:
    # Heuristic fallback: shorter input → shorter expected output.
    # The court-side run produced ~150-300 output tokens for ~600-char inputs;
    # treat short bucket as 200, mid as 350, long as 500.
    for b in merged:
        if b['prompt_p99'] < 1700:        rec = 256
        elif b['prompt_p99'] < 1900:      rec = 320
        elif b['prompt_p99'] < 2200:      rec = 384
        else:                              rec = 480
        b['out_p50'] = b['out_p99'] = b['out_max'] = -1  # unknown
        b['rec_max_new_tokens'] = rec
    print('Using heuristic fallback for rec_max_new_tokens — please run the GPU pilot for accurate values.')

# Re-tighten max_model_len now that we know real per-bucket output budgets.
for b in merged:
    b['rec_max_model_len'] = round_up(b['prompt_p99'] + b['rec_max_new_tokens'] + MAX_MODEL_LEN_HEADROOM,
                                       MAX_MODEL_LEN_ROUND)
    per_slot_tokens = b['rec_max_model_len']
    raw_slots = kv_pool_tokens // per_slot_tokens
    b['rec_max_num_seqs'] = int(min(raw_slots, MAX_NUM_SEQS_CAP))
    b['per_slot_kv_mib'] = round(per_slot_tokens * PER_TOKEN_KV_BYTES / 1024**2, 2)

df_buckets = pd.DataFrame(merged)
print(df_buckets[['lo','hi','count','rec_max_text_chars','rec_max_new_tokens',
                  'rec_max_model_len','rec_max_num_seqs']].to_string(index=False))

## Cell 10 — Throughput model: bucketed vs. baseline

Rough first-order model only. Takes the per-bucket (count × output_p50) as the dominant tokens-to-decode and divides by `decode_tps_per_slot × max_num_seqs`. We use the user's empirical baseline of ~60–90 min for 173k rows with 384 slots and ~150 tok/output to back out per-slot decode TPS, then re-apply per bucket.

Caveats:
- vLLM's continuous batching already smooths some of the variance the buckets exploit.
- Per-slot decode TPS isn't constant; it drops at very high concurrency due to attention contention.
- Engine reload between buckets has its own ~2–3 min cost, factored in below.

In [ ]:
# --- knobs (override with your real measurements) ---
BASELINE_RUNTIME_MIN = 75            # observed wall-clock of the current single-config run
BASELINE_MAX_NUM_SEQS = 384
BASELINE_AVG_OUT_TOKENS = 200        # rough: median JSON output
ENGINE_RELOAD_MIN = 2.5              # vLLM init cost per bucket
TOTAL_ROWS = len(df)

baseline_total_out_tokens = TOTAL_ROWS * BASELINE_AVG_OUT_TOKENS
baseline_decode_seconds = BASELINE_RUNTIME_MIN * 60
decode_tps_total = baseline_total_out_tokens / baseline_decode_seconds
decode_tps_per_slot = decode_tps_total / BASELINE_MAX_NUM_SEQS
print(f'Calibrated decode TPS: {decode_tps_total:,.0f} total, '
      f'{decode_tps_per_slot:.1f} per-slot at 384 slots.')

# Bucketed throughput estimate.
total_min_bucketed = 0.0
for b in merged:
    out_tokens_avg = b.get('out_p50') if b.get('out_p50', -1) > 0 else BASELINE_AVG_OUT_TOKENS
    bucket_total_out_tokens = b['count'] * out_tokens_avg
    # Effective TPS scales sublinearly with slots — use square-root attenuation past 384.
    eff_slots = b['rec_max_num_seqs']
    if eff_slots > BASELINE_MAX_NUM_SEQS:
        eff_slots = BASELINE_MAX_NUM_SEQS + (eff_slots - BASELINE_MAX_NUM_SEQS) * 0.6
    bucket_decode_seconds = bucket_total_out_tokens / (decode_tps_per_slot * eff_slots)
    bucket_min = bucket_decode_seconds / 60 + ENGINE_RELOAD_MIN
    b['est_runtime_min'] = round(bucket_min, 2)
    total_min_bucketed += bucket_min

# Also estimate the simpler 'one engine, per-row max_tokens' approach: NO engine reload, NO
# concurrency change, just per-row max_tokens=bucket_max_new_tokens. Saves the wasted-tail cost
# only on rows whose model produced > bucket_max but < global 450 (rare but real).
single_engine_min = (TOTAL_ROWS * BASELINE_AVG_OUT_TOKENS) / (decode_tps_per_slot * BASELINE_MAX_NUM_SEQS) / 60

print(f'\n--- Runtime estimate ---')
print(f'  Baseline (current single-config)         : {BASELINE_RUNTIME_MIN:>5.1f} min  (measured)')
print(f'  Single engine + per-row max_tokens caps  : {single_engine_min:>5.1f} min  (lower bound)')
print(f'  Multi-engine bucketed run (incl. reload) : {total_min_bucketed:>5.1f} min')
print(f'  Multi-engine reload overhead             : {ENGINE_RELOAD_MIN * len(merged):>5.1f} min')
print(f'\nVerdict (rough): bucketing is most worthwhile if the long-tail bucket is large enough')
print(f'that its higher max_num_seqs gain outweighs the {len(merged)} × {ENGINE_RELOAD_MIN}-min reload tax.')

## Cell 11 — Write the recommended config to JSON

In [ ]:
config_out = {
    'meta': {
        'generated_from': str(INPUT_PATH),
        'total_rows': int(TOTAL_ROWS),
        'sample_size': int(len(sample_df)),
        'tokenizer': MODEL_NAME,
        'system_prefix_tokens': int(PROBE_LEN),
        'kv_dtype_bytes': KV_DTYPE_BYTES,
        'kv_pool_tokens_estimated': int(kv_pool_tokens),
        'pilot_ran': bool(PILOT_RAN),
    },
    'buckets': [
        {
            'id': i,
            'prompt_tok_lo': b['lo'],
            'prompt_tok_hi': b['hi'],
            'row_count': b['count'],
            'prompt_p50': b['prompt_p50'],
            'prompt_p99': b['prompt_p99'],
            'prompt_max': b['prompt_max'],
            'text_p50': b['text_p50'],
            'text_p99': b['text_p99'],
            'text_max': b['text_max'],
            'out_p50': b['out_p50'],
            'out_p99': b['out_p99'],
            'out_max': b['out_max'],
            'rec_max_text_chars': b['rec_max_text_chars'],
            'rec_max_new_tokens': b['rec_max_new_tokens'],
            'rec_max_model_len': b['rec_max_model_len'],
            'rec_max_num_seqs': b['rec_max_num_seqs'],
            'per_slot_kv_mib': b['per_slot_kv_mib'],
            'est_runtime_min': b.get('est_runtime_min'),
        }
        for i, b in enumerate(merged)
    ],
}

with CONFIG_OUT.open('w', encoding='utf-8') as f:
    json.dump(config_out, f, indent=2, ensure_ascii=False)

print(f'Wrote bucketed config: {CONFIG_OUT}')
print(json.dumps(config_out['meta'], indent=2))
print(f'{len(config_out["buckets"])} buckets recorded.')

## Cell 12 — Recommendation summary

Once this notebook has been run on real data, fill in this table by hand for the writeup. Three actions to consider, in order of cost:

1. **Cheap win — per-row `max_tokens` (no engine change).**
   Pass `SamplingParams(max_tokens=bucket['rec_max_new_tokens'])` per row in the existing `cfg.submit_chunk` call. No reload, no config gymnastics. Saves time on degenerate-output rows where the model otherwise generates up to 450 tokens of garbage before hitting the ceiling. **Recommended unconditionally.**

2. **Medium win — split into 2–3 engine runs by length.**
   E.g. `short` (text < 300 chars), `medium` (300–1500), `long` (≥1500, no truncation). Three engine loads cost ~7 min; the short engine can run with `max_model_len ≈ 1900, max_num_seqs ≈ 700` which roughly doubles concurrency for the 60% short-text majority. Worth it if baseline is > 60 min.

3. **Expensive win — full 17-bucket plan.**
   Generally not worth the reload tax (≥40 min in setup alone) unless the bulk of the corpus is concentrated in 2–3 consecutive buckets that already merge. Run cell 6 first; if `len(merged) ≤ 4` after merging, it's the same as option 2.

The numbers in `laws_de_bucketed_config.json` tell you which option this corpus actually justifies. If the heuristic fallback was used (no GPU pilot), the `rec_max_new_tokens` values are conservative — re-run with `RUN_GPU_PILOT = True` on the Kaggle GPU before basing a production change on them.